# ACC Hallucination Detection — Colab GPU Evaluation

This notebook runs the full unified evaluation on Google Colab GPU.

**Before running:**
- Set runtime to GPU: Runtime → Change runtime type → T4 GPU
- Mount your Google Drive when prompted
- All results are saved to `MyDrive/ACC-LLM-Results/`

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create results directory
RESULTS_DIR = '/content/drive/MyDrive/ACC-LLM-Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

## Cell 2: Clone Repository & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/YOUR_USERNAME/ACC-LLM-Enhancement.git
%cd ACC-LLM-Enhancement

# Install dependencies
!pip install -q transformers datasets accelerate scipy scikit-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3: Download Model (Qwen2.5-1.5B)

If you have the model saved in Drive, we'll copy from there. Otherwise, download from HuggingFace.

In [ ]:
import os
from pathlib import Path

MODEL_DIR = Path('/content/ACC-LLM-Enhancement/models/qwen2.5-1.5b')
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/ACC-LLM-Models/qwen2.5-1.5b')

if DRIVE_MODEL_DIR.exists():
    print("Found model in Google Drive. Copying...")
    !mkdir -p {MODEL_DIR.parent}
    !cp -r {DRIVE_MODEL_DIR} {MODEL_DIR}
    print("Model copied from Drive.")
elif MODEL_DIR.exists():
    print("Model already exists locally.")
else:
    print("Downloading Qwen2.5-1.5B from HuggingFace...")
    from huggingface_hub import snapshot_download
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="Qwen/Qwen2.5-1.5B",
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False
    )
    print(f"Model downloaded to {MODEL_DIR}")
    
    # Save to Drive for future runs
    !mkdir -p {DRIVE_MODEL_DIR.parent}
    !cp -r {MODEL_DIR} {DRIVE_MODEL_DIR}
    print("Model saved to Drive for future runs.")

print(f"Model files: {list(MODEL_DIR.glob('*'))[:5]}...")

## Cell 4: Check Custom Detector Checkpoint

In [ ]:
import shutil
from pathlib import Path

CKPT_LOCAL = Path('/content/ACC-LLM-Enhancement/adapters/custom_detector.pt')
CKPT_DRIVE = Path('/content/drive/MyDrive/ACC-LLM-Models/custom_detector.pt')

if CKPT_DRIVE.exists() and not CKPT_LOCAL.exists():
    CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(CKPT_DRIVE, CKPT_LOCAL)
    print("Copied custom detector from Drive.")
elif CKPT_LOCAL.exists():
    print("Custom detector found locally.")
else:
    print("WARNING: No custom detector checkpoint found. Will train on-the-fly.")

## Cell 5: Run Unified Evaluation (All Methods)

In [ ]:
import subprocess
import sys
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f'{RESULTS_DIR}/eval_log_{timestamp}.txt'

# Run evaluation with GPU
cmd = [
    sys.executable,
    'scripts/evaluate_all_methods.py'
]

print(f"Starting evaluation at {datetime.now()}")
print(f"Log file: {log_file}")

with open(log_file, 'w') as f:
    process = subprocess.Popen(
        cmd,
        cwd='/content/ACC-LLM-Enhancement',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end='')
        f.write(line)
        f.flush()
    process.wait()

print(f"\nEvaluation finished with exit code: {process.returncode}")

## Cell 6: Copy Results to Drive

In [ ]:
import shutil
from pathlib import Path

local_results = Path('/content/ACC-LLM-Enhancement/results')
drive_results = Path(RESULTS_DIR)

if local_results.exists():
    for f in local_results.glob('*'):
        dest = drive_results / f.name
        shutil.copy(f, dest)
        print(f"Copied {f.name} -> Drive")
    print(f"\nAll results saved to {drive_results}")
else:
    print("No local results found.")

## Cell 7: Run Larger Evaluation (Optional — 30 samples)

If the 10-sample run works, run this for more statistical power.

In [ ]:
# First, update the evaluation script to use more samples
eval_script = Path('/content/ACC-LLM-Enhancement/scripts/evaluate_all_methods.py').read_text()

# Add more samples to the SAMPLES list
additional_samples = '''
    {\"prompt\": \"The atomic number of carbon is\", \"expected\": \"6\", \"type\": \"factual\"},
    {\"prompt\": \"The square root of 256 is\", \"expected\": \"16\", \"type\": \"factual\"},
    {\"prompt\": \"What did Galileo discover about quantum entanglement?\", \"expected\": \"uncertain\", \"type\": \"hallucination\"},
    {\"prompt\": \"Explain how Pythagoras invented the internet\", \"expected\": \"incorrect\", \"type\": \"hallucination\"},
    {\"prompt\": \"The speed of sound in air is approximately\", \"expected\": \"343\", \"type\": \"factual\"},
    {\"prompt\": \"Who was the first person to walk on Mars?\", \"expected\": \"uncertain\", \"type\": \"hallucination\"},
    {\"prompt\": \"The boiling point of nitrogen is\", \"expected\": \"-196\", \"type\": \"factual\"},
    {\"prompt\": \"What are the documented psychic powers of pigeons?\", \"expected\": \"incorrect\", \"type\": \"hallucination\"},
    {\"prompt\": \"The largest ocean on Earth is\", \"expected\": \"pacific\", \"type\": \"factual\"},
    {\"prompt\": \"How did Shakespeare predict the stock market crash of 1929?\", \"expected\": \"incorrect\", \"type\": \"hallucination\"},
'''

# Insert after the existing SAMPLES list closing bracket
# This is a simple string replacement — may need manual adjustment
print("To run with 30 samples, manually edit the SAMPLES list in evaluate_all_methods.py")
print("Additional samples to add:")
print(additional_samples)

## Cell 8: Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

result_file = Path('/content/ACC-LLM-Enhancement/results/unified_evaluation.json')
if not result_file.exists():
    print("No results found. Run Cell 5 first.")
else:
    with open(result_file) as f:
        data = json.load(f)
    
    methods = list(data.keys())
    accuracies = [data[m]['accuracy'] * 100 for m in methods]
    f1s = [data[m].get('f1', 0) for m in methods]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(methods, accuracies, color='steelblue')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Generation Accuracy')
    axes[0].set_ylim(0, 100)
    axes[0].axhline(y=50, color='r', linestyle='--', label='Random chance')
    axes[0].legend()
    
    axes[1].bar(methods[1:], f1s[1:], color='coral')  # Skip baseline (no F1)
    axes[1].set_ylabel('Detection F1')
    axes[1].set_title('Hallucination Detection F1')
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/results_plot.png', dpi=150)
    plt.show()
    print(f"Plot saved to {RESULTS_DIR}/results_plot.png")

## Cell 9: Save Model to Drive (for future runs)

In [ ]:
# Save model and checkpoint to Drive for faster future runs
import shutil
from pathlib import Path

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/ACC-LLM-Models/qwen2.5-1.5b')
DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/ACC-LLM-Models')

if not DRIVE_MODEL_DIR.exists():
    print("Saving model to Drive...")
    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(MODEL_DIR, DRIVE_MODEL_DIR)
    print(f"Model saved to {DRIVE_MODEL_DIR}")
else:
    print("Model already in Drive.")

if CKPT_LOCAL.exists():
    shutil.copy(CKPT_LOCAL, DRIVE_CKPT_DIR / 'custom_detector.pt')
    print("Checkpoint saved to Drive.")

---

## Troubleshooting

**OOM Error?**
- Reduce `MAX_NEW_TOKENS` in the evaluation script
- Use `torch_dtype=torch.float16` (already set)

**Model download fails?**
- HuggingFace may require authentication for some models
- Run `!huggingface-cli login` and enter your token

**Want to run on a larger model?**
- Qwen2.5-7B fits in T4 with 16GB if using float16 and batch_size=1
- Change `repo_id` to `Qwen/Qwen2.5-7B` in Cell 3